In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_timestamp, avg, stddev, lag, round,
    when, row_number, unix_timestamp, lit, to_date, current_timestamp,expr,from_json
)
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from datetime import datetime
import traceback
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("KafkaCryptoConsumer_Idempotent") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.executor.memory", "1g") \
    .config("spark.executor.cores", "1") \
    .config("spark.cores.max", "2") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()


# -----------------------------
# Schema
# -----------------------------
coin_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_time", StringType(), True),
    StructField("producer_time", StringType(), True),
    StructField("producer_id", StringType(), True),
    StructField("id", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("current_price", DoubleType(), True),
    StructField("market_cap", DoubleType(), True),
    StructField("total_volume", DoubleType(), True),
    StructField("high_24h", DoubleType(), True),
    StructField("low_24h", DoubleType(), True),
    StructField("last_updated", StringType(), True),
])

spark.sparkContext.setLogLevel("WARN")

# -----------------------------
# Create Delta Table (ONE TIME SAFE)
# -----------------------------
spark.sql("""
CREATE TABLE IF NOT EXISTS crypto_bronze (
    event_id STRING,
    event_time TIMESTAMP,
    producer_time TIMESTAMP,
    producer_id STRING,
    id STRING,
    symbol STRING,
    current_price DOUBLE,
    market_cap DOUBLE,
    total_volume DOUBLE,
    high_24h DOUBLE,
    low_24h DOUBLE,
    last_updated STRING,
    kafka_timestamp TIMESTAMP,
    partition INT,
    offset LONG,
    event_date DATE
)
USING DELTA
PARTITIONED BY (event_date)
LOCATION 's3a://crypto-data/bronze/'
""")

# -----------------------------
# Kafka Stream
# -----------------------------
raw_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka-1:9092,kafka-2:9092,kafka-3:9092") \
    .option("subscribe", "crypto-prices") \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .load()

# -----------------------------
# Convert binary → string
# -----------------------------
json_df = raw_df.select(
    col("value").cast("string").alias("json"),
    col("timestamp").alias("kafka_timestamp"),
    col("partition"),
    col("offset")
)

# -----------------------------
# Parse JSON
# -----------------------------
parsed_df = json_df.select(
    from_json(col("json"), coin_schema).alias("coin"),
    "kafka_timestamp",
    "partition",
    "offset"
)

# -----------------------------
# Flatten + enrich
# -----------------------------
flattened_df = parsed_df.select(
    "coin.*",
    "kafka_timestamp",
    "partition",
    "offset"
).withColumn(
    "event_time",
    to_timestamp("event_time", "yyyy-MM-dd'T'HH:mm:ss.SSSSSSX")
).withColumn(
    "producer_time",
    to_timestamp("producer_time", "yyyy-MM-dd'T'HH:mm:ss.SSSSSSX")
).withColumn(
    "event_date",
    to_date("event_time")
)

flattened_df = flattened_df.dropDuplicates(["event_id"])

# -----------------------------
# Idempotent UPSERT Logic
# -----------------------------
def upsert_to_delta(batch_df, batch_id):

    if batch_df.isEmpty():
        return

    batch_df.createOrReplaceTempView("updates")

    merge_sql = """
    MERGE INTO crypto_bronze t
    USING updates s
    ON t.event_id = s.event_id

    WHEN MATCHED THEN UPDATE SET *

    WHEN NOT MATCHED THEN INSERT *
    """

    # IMPORTANT: use same Spark session
    batch_df.sparkSession.sql(merge_sql)

# -----------------------------
# Write Stream (Idempotent)
# -----------------------------
query = flattened_df.writeStream \
    .foreachBatch(upsert_to_delta) \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://crypto-data/checkpoints/kafka_to_delta/") \
    .trigger(processingTime="30 seconds") \
    .start()

query.awaitTermination()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 12:48:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/06 12:49:04 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/05/06 12:49:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/06 12:49:24 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
ERROR:root:Exception while sending command.                                     
Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.str

Py4JError: An error occurred while calling o101.awaitTermination

In [ ]:
query.stop()

In [ ]:
spark